# Face Blur Batch Processor

This notebook runs the automated face-blurring pipeline over all videos in `gs://tbd-qa/for_fb`.

**How it works**
1. Discovers all `.mp4` / `.mov` files under the input GCS prefix.
2. Skips any video that already has a `success` record in the manifest.
3. Downloads each video, detects/tracks faces, blurs them, re-uploads.
4. Writes a summary to the manifest at `gs://tbd-qa/post_fb/processed_manifest.jsonl`.

**Safe to re-run** — already-processed videos are automatically skipped.

---
## Step 1 — Set up environment

In [1]:
# Install system dependencies (ffmpeg for video muxing)
!apt-get update -qq && apt-get install -y -qq ffmpeg libgl1

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [2]:
# Install Python dependencies
!pip install -q \
    google-cloud-storage \
    opencv-python-headless \
    numpy \
    scipy \
    insightface \
    onnxruntime \
    mediapipe

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 439.5/439.5 kB 8.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 65.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 71.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 85.1 MB/s eta 0:00:00


## Step 2 — Authenticate to Google Cloud

### Option A: Interactive user auth (simplest for Colab)
This opens a browser window to log in with your Google account.  
Your account needs Storage Object Viewer on `tbd-qa` (input) and  
Storage Object Creator / Admin on the output prefix.

In [17]:
from google.colab import auth
auth.authenticate_user()
project_id = 'gdmdelivery'
!gcloud config set project {project_id}
!gsutil ls
print("✅  Authenticated with Google user account")

[environment: untagged] Read more g.co/cloud/project-env-tag.
Updated property [core/project].
Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
gs://gdmdelivery_cloudbuild/
gs://monologue_video_data/
gs://ready_for_review/
gs://tbd-qa/
gs://tbd_uploads/
gs://xr_data/
✅  Authenticated with Google user account


In [18]:
# Project ID set above — expose as variable for later cells
GCP_PROJECT = "gdmdelivery"

### Option B: Service account JSON key
Upload your service account key JSON to Colab, then run:

```python
import os
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/content/my-service-account.json"
```

⚠️ Do not commit service account keys to git.

In [19]:
# Confirm bucket access
from google.cloud import storage

client = storage.Client(project=GCP_PROJECT)
blobs = list(client.list_blobs("tbd-qa", prefix="for_fb", max_results=5))
print(f"✅  Bucket access confirmed — first {len(blobs)} blob(s):")
for b in blobs:
    print(f"    gs://tbd-qa/{b.name}")

✅  Bucket access confirmed — first 2 blob(s):
    gs://tbd-qa/for_fb/
    gs://tbd-qa/for_fb/031426_CG_Data_CG_PID70_2802_CG_PID70_2802.mov


## Step 3 — Load the project code

**Option A: Clone from GitHub** (recommended)
```python
!git clone https://github.com/YOUR_ORG/face_blur_tool.git
%cd face_blur_tool
```

**Option B: Upload a zip**  
1. Zip the `face_blur_tool/` directory on your machine.
2. Upload it via the Colab file browser (left sidebar → Upload).
3. Run the cell below to unzip.

In [ ]:
import os, sys

# ── Adjust this path to wherever you cloned / uploaded the project ──
PROJECT_ROOT = "/content/face_blur_tool/face_blur_tool"   # change if needed

# If src is one level deeper (common when repo is zipped with an extra folder),
# adjust PROJECT_ROOT automatically
if os.path.isdir(os.path.join(PROJECT_ROOT, "face_blur_tool", "src")):
    PROJECT_ROOT = os.path.join(PROJECT_ROOT, "face_blur_tool")

# Ensure PROJECT_ROOT is at the front of sys.path
if PROJECT_ROOT in sys.path:
    sys.path.remove(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

# Confirm the path exists
print("Project root exists:", os.path.isdir(PROJECT_ROOT))
print("sys.path[0]:", sys.path[0])

# Quick sanity check — import the config module
from src.config import FaceBlurConfig
print("✅  Project code loaded successfully")

In [9]:
import os
from google.cloud import storage

# Ensure the local project root directory exists
os.makedirs(PROJECT_ROOT, exist_ok=True)

client = storage.Client(project=GCP_PROJECT)
bucket_name = "tbd-qa"
prefix = "scripts/face_blur_tool/"

print(f"Downloading files from gs://{bucket_name}/{prefix} to {PROJECT_ROOT}")

blobs = client.list_blobs(bucket_name, prefix=prefix)
downloaded_count = 0
for blob in blobs:
    # Construct the local file path, preserving subdirectory structure
    # Skip the prefix itself if it's treated as a folder (common for GCS folders)
    relative_path = os.path.relpath(blob.name, prefix)
    if relative_path == ".": # This handles the case where blob.name is exactly the prefix (e.g., 'scripts/face_blur_tool/')
        continue

    local_file_path = os.path.join(PROJECT_ROOT, relative_path)

    # Ensure the local subdirectory exists
    os.makedirs(os.path.dirname(local_file_path), exist_ok=True)

    # Download the file
    if not blob.name.endswith('/'): # Skip downloading directory placeholders
        print(f"Downloading {blob.name} to {local_file_path}")
        blob.download_to_filename(local_file_path)
        downloaded_count += 1

print(f"✅ Downloaded {downloaded_count} files to {PROJECT_ROOT}")

NameError: name 'GCP_PROJECT' is not defined

In [10]:
import os
import zipfile

zip_file_path = '/content/face_blur_tool.zip'
output_dir = PROJECT_ROOT

if os.path.exists(zip_file_path):
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall(output_dir)
    print(f"✅ Successfully unzipped {zip_file_path} to {output_dir}/")
else:
    print(f"⚠️ Zip file not found at {zip_file_path}. Please ensure it has been uploaded.")

✅ Successfully unzipped /content/face_blur_tool.zip to /content/face_blur_tool/


In [ ]:
# After unzipping, let's re-run the check to confirm the project root exists and import the config module.
# This is a re-run of cells 4pgZ_707F8ci and Kc-HfVpjF8ci.

import os, sys

# The PROJECT_ROOT is already defined from a previous cell

# Ensure PROJECT_ROOT is at the front of sys.path, re-adding it if necessary
if PROJECT_ROOT in sys.path:
    sys.path.remove(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

# Don't chdir — just confirm the path exists
print("Project root exists:", os.path.isdir(PROJECT_ROOT))
print("sys.path[0]:", sys.path[0])

# Quick sanity check — import the config module
from src.config import FaceBlurConfig
print("✅  Project code loaded successfully")

Project root exists: True
sys.path[0]: /content/face_blur_tool/face_blur_tool
✅  Project code loaded successfully


In [15]:
# After downloading, let's re-run the check to confirm the project root exists and import the config module.
# This is a re-run of cells 4pgZ_707F8ci and Kc-HfVpjF8ci.

import os, sys

# The PROJECT_ROOT is already defined from a previous cell

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Don't chdir — just confirm the path exists
print("Project root exists:", os.path.isdir(PROJECT_ROOT))
print("sys.path[0]:", sys.path[0])

# Quick sanity check — import the config module
from src.config import FaceBlurConfig
print("✅  Project code loaded successfully")

Project root exists: True
sys.path[0]: /content/face_blur_tool/face_blur_tool
✅  Project code loaded successfully


In [16]:
# Quick sanity check — import the config module
from src.config import FaceBlurConfig
print("✅  Project code loaded successfully")

✅  Project code loaded successfully


## Step 4 — Configure the pipeline

In [20]:
from src.config import FaceBlurConfig

config = FaceBlurConfig(
    # ── GCS paths (defaults match the project spec) ──────────────────────
    input_prefix   = "gs://tbd-qa/for_fb",
    output_prefix  = "gs://tbd-qa/post_fb",
    manifest_path  = "gs://tbd-qa/post_fb/processed_manifest.jsonl",

    # ── Detector ─────────────────────────────────────────────────────────
    # "retinaface" (recommended) or "mediapipe" (faster, lower recall)
    detector_type               = "retinaface",
    detector_confidence_threshold = 0.35,   # low → bias toward recall
    min_face_size               = 30,       # pixels; ignore tiny faces

    # ── Sampling ─────────────────────────────────────────────────────────
    detection_interval = 4,   # run detector every N frames; tracker fills in between

    # ── Tiled detection (improves recall on small/background faces) ──────
    use_tiled_detection = True,
    tile_size           = 640,   # px — each tile passed to the detector
    tile_overlap        = 0.25,  # 25% overlap between adjacent tiles
    tile_nms_threshold  = 0.4,   # IoU threshold to suppress duplicate detections

    # ── Blur ─────────────────────────────────────────────────────────────
    blur_padding_ratio = 0.35,   # expand bbox by 35% on each side
    blur_strength      = 25.0,   # Gaussian sigma

    # ── Other ─────────────────────────────────────────────────────────────
    preserve_audio = True,
    local_temp_dir = "/tmp/face_blur",
    gcp_project    = GCP_PROJECT,
)

print("Config:")
print(f"  input_prefix  : {config.input_prefix}")
print(f"  output_prefix : {config.output_prefix}")
print(f"  manifest_path : {config.manifest_path}")
print(f"  detector_type : {config.detector_type}")

Config:
  input_prefix  : gs://tbd-qa/for_fb
  output_prefix : gs://tbd-qa/post_fb
  manifest_path : gs://tbd-qa/post_fb/processed_manifest.jsonl
  detector_type : retinaface


## Step 5 — (Optional) Dry run — list videos without processing

In [21]:
from src.pipeline.batch_runner import run_batch

print("Dry-run discovery:")
run_batch(client, config, dry_run=True)

Dry-run discovery:
23:20:51 [INFO] src.manifest – Loaded 0 manifest records from GCS
23:20:51 [INFO] src.video_inventory – Discovered 1 video(s) under gs://tbd-qa/for_fb
23:20:51 [INFO] src.pipeline.batch_runner – Dry run — would process 1 video(s):
  gs://tbd-qa/for_fb/031426_CG_Data_CG_PID70_2802_CG_PID70_2802.mov


BatchSummary(total_discovered=1, skipped=0, succeeded=0, failed=0, results=[])

## Step 6 — Run the batch processor

> ⚠️ This will download, process, and re-upload videos. Make sure your config is correct before running.
>
> Already-successful videos (per manifest) will be **skipped automatically**.

In [22]:
from src.logging_utils import configure_root_logger

# Set to "DEBUG" for verbose output during debugging
configure_root_logger(level="INFO")

summary = run_batch(client, config)

23:21:05 [INFO] src.manifest – Loaded 0 manifest records from GCS
23:21:05 [INFO] src.video_inventory – Discovered 1 video(s) under gs://tbd-qa/for_fb
23:21:05 [INFO] src.pipeline.batch_runner – Initialising detector: retinaface
download_path: /root/.insightface/models/buffalo_sc


100%|██████████| 14619/14619 [00:00<00:00, 64422.67KB/s]


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_sc/det_500m.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_sc/w600k_mbf.onnx recognition ['None', 3, 112, 112] 127.5 127.5
set det-size: (640, 640)
23:21:11 [INFO] src.detectors.retinaface_detector – RetinaFaceDetector loaded model=buffalo_sc providers=['CPUExecutionProvider']
23:21:11 [INFO] src.pipeline.batch_runner – [1/1] Processing gs://tbd-qa/for_fb/031426_CG_Data_CG_PID70_2802_CG_PID70_2802.mov
23:21:11 [INFO] src.pipeline.process_video – Starting video processing
23:21:11 [INFO] src.pipeline.process_video – Downloading gs://tbd-qa/for_fb/031426_CG_Data_CG_PID70_2802_CG_PID70_2802.mov → /tmp/face_blur/tmpcrsh8a9v.mov
23:21:11 [INFO] src.gcs_io – Downloading gs://tbd-qa/for_fb/031426_CG_Data_CG_PID70_2802_CG_PID70_2802.

## Step 7 — Results summary

In [23]:
summary.print_summary()

print("\nOutput locations:")
for r in summary.results:
    if r.succeeded:
        print(f"  ✅  {r.output_gcs_path}")
    elif r.failed:
        print(f"  ❌  {r.source_gcs_path} → {r.error_message}")
    else:
        print(f"  ⏭  {r.source_gcs_path} (skipped)")


  FACE BLUR BATCH SUMMARY
  Discovered : 1
  Skipped    : 0  (already processed)
  Succeeded  : 1
  Failed     : 0


Output locations:
  ✅  gs://tbd-qa/post_fb/031426_CG_Data_CG_PID70_2802_CG_PID70_2802_blurred.mov


## Step 8 — Inspect the manifest

In [24]:
import json
from src import gcs_io
from src.utils.paths import gcs_uri_to_bucket_and_blob

bucket, blob = gcs_uri_to_bucket_and_blob(config.manifest_path)
lines = gcs_io.read_jsonl_blob(client, bucket, blob)

print(f"Manifest contains {len(lines)} record(s):")
for line in lines[-20:]:   # show last 20
    rec = json.loads(line)
    status_icon = "✅" if rec["status"] == "success" else "❌"
    print(f"  {status_icon}  {rec['source_gcs_path']}  →  {rec['status']}")

Manifest contains 1 record(s):
  ✅  gs://tbd-qa/for_fb/031426_CG_Data_CG_PID70_2802_CG_PID70_2802.mov  →  success


## Troubleshooting

| Problem | Fix |
|---------|-----|
| `ModuleNotFoundError: src` | Make sure `PROJECT_ROOT` is in `sys.path` (Step 2) |
| `ffprobe not found` | Run `!apt-get install -y ffmpeg` again |
| `google.auth.exceptions.DefaultCredentialsError` | Re-run the auth cell (Step 3) |
| `403 Forbidden` on GCS | Your account needs Storage Object roles on `tbd-qa` |
| `insightface` install fails | Try `!pip install insightface onnxruntime` with runtime restart |
| Video fails with codec error | Confirm `ffmpeg` is installed; check the error message in the manifest |
| Want to reprocess a video | Delete its manifest line, or set `status=failed` — it will be retried |
| Runtime disconnects mid-batch | Re-run the batch cell — already-successful videos are skipped |
